In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Parse s{seed}.log files, extract the *last* block after "start to eval",
then aggregate metrics across the provided seeds and save mean/std to result.csv.

Expected log snippets (order matters within an eval block):
start to eval
hit20 <float>
hit50 <float>
hit100 <float>
roc_auc, pr_auc, f1, mrr <float> <float> <float> <float>

Usage:
  python parse_eval_logs.py --seeds 388 389 390 --dir . --pattern "s{seed}.log"

Notes:
- Only the last "start to eval" block in each file is used.
- Files missing a complete eval block are skipped (with a warning).
- Output: result.csv in the chosen directory.
"""
import argparse
import os
import re
import sys
from typing import Dict, List, Optional

import pandas as pd

HIT_PAT = re.compile(r'^hit(20|50|100)\s+([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s*$')
METRICS_PAT = re.compile(
    r'^roc_auc,\s*pr_auc,\s*f1,\s*mrr\s+'
    r'([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+'
    r'([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+'
    r'([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+'
    r'([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s*$'
)

METRIC_KEYS = ['hit20', 'hit50', 'hit100', 'roc_auc', 'pr_auc', 'f1', 'mrr']


def extract_last_eval_block(text: str) -> Optional[Dict[str, float]]:
    lines = text.splitlines()
    last_block: Optional[Dict[str, float]] = None
    i = 0
    n = len(lines)
    while i < n:
        if 'start to eval' in lines[i]:
            tmp: Dict[str, float] = {}
            j = i + 1
            # Parse forward until we hit the metrics line or EOF
            while j < n:
                line = lines[j].strip()
                m = HIT_PAT.match(line)
                if m:
                    key = f"hit{m.group(1)}"
                    try:
                        tmp[key] = float(m.group(2))
                    except ValueError:
                        pass
                    j += 1
                    continue

                m2 = METRICS_PAT.match(line)
                if m2:
                    try:
                        tmp['roc_auc'] = float(m2.group(1))
                        tmp['pr_auc'] = float(m2.group(2))
                        tmp['f1'] = float(m2.group(3))
                        tmp['mrr'] = float(m2.group(4))
                    except ValueError:
                        pass
                    # We consider this block complete; record as "last"
                    last_block = tmp
                    i = j  # move outer pointer forward
                    break
                j += 1
        i += 1
    return last_block


def parse_one_file(path: str) -> Optional[Dict[str, float]]:
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"[WARN] File not found: {path}", file=sys.stderr)
        return None
    except Exception as e:
        print(f"[WARN] Failed to read {path}: {e}", file=sys.stderr)
        return None

    block = extract_last_eval_block(content)
    if block is None or not all(k in block for k in ['roc_auc', 'pr_auc', 'f1', 'mrr']):
        # Minimal requirement: the summary metrics line must be present
        print(f"[WARN] No complete eval block found in {path}", file=sys.stderr)
        return None
    # Fill missing hits (if any) with NaN for aggregation
    for k in ['hit20', 'hit50', 'hit100']:
        block.setdefault(k, float('nan'))
    return block


def main():
    seeds = [1, 2, 3, 4, 5]
    dir = "."
    pattern = "s{seed}-r0.02-e100.log"
    outfile = "result-e100.csv"

    rows = []
    for seed in seeds:
        fname = pattern.format(seed=seed)
        path = os.path.join(dir, fname)
        block = parse_one_file(path)
        if block is None:
            continue
        row = {'seed': seed}
        row.update({k: block.get(k, float('nan')) for k in METRIC_KEYS})
        rows.append(row)

    if not rows:
        print("[ERROR] No valid logs parsed. Exiting.", file=sys.stderr)
        sys.exit(2)

    df = pd.DataFrame(rows).set_index('seed').sort_index()

    # Compute mean ± std per metric across the available seeds (skip NaN)
    summary = []
    for k in METRIC_KEYS:
        series = df[k]
        mean = series.mean(skipna=True) * 100
        std = series.std(skipna=True, ddof=1) * 100
        n = int(series.count())
        summary.append({
            'metric': k,
            'mean_std': f"{mean:.2f} ± {std:.2f}",
            'n': n
        })

    out = pd.DataFrame(summary, columns=['metric', 'mean_std', 'n'])

    out_path = os.path.join(dir, outfile)
    out.to_csv(out_path, index=False)
    print(f"[OK] Wrote summary to {out_path}")
    # Optional: also show per-seed table path if user needs it (commented out)
    # df.to_csv(os.path.join(args.dir, 'per_seed.csv'))


if __name__ == '__main__':
    main()


[OK] Wrote summary to ./result-e100.csv


[WARN] File not found: ./s2-r0.02-e100.log
[WARN] File not found: ./s3-r0.02-e100.log
[WARN] File not found: ./s4-r0.02-e100.log
[WARN] File not found: ./s5-r0.02-e100.log
